### INGESTA DE DATOS A LA CAPA RAW

**CONEXIÓN CON LA API DE SPOTIFY**

In [0]:
import requests
import base64
import time
from datetime import datetime
import json

In [0]:
client_id = "xxxxxxxxxxx"
client_secret = "xxxxxxxxxxxx"

In [0]:
# OBTENER EL TOKEN DE ACCESO (FORMA 1)

url_token = "https://accounts.spotify.com/api/token"

response = requests.post(
    url_token,
    data={
        "grant_type": "client_credentials"
    },
    auth=(
        client_id,
        client_secret
    )
)

token = response.json()["access_token"]
print("Token obtenido correctamente")

In [0]:
# OBTENER EL TOKEN DE ACCESO (FORMA 2)

client_id = dbutils.secrets.get(
    "spotify",
    "spotify-client-id"
)

client_secret = dbutils.secrets.get(
    "spotify",
    "spotify-client-secret"
)

credentials = f"{client_id}:{client_secret}"

encoded_credentials = base64.b64encode(
    credentials.encode()
).decode()

response = requests.post(
    "https://accounts.spotify.com/api/token",
    headers={
        "Authorization": f"Basic {encoded_credentials}",
        "Content-Type": "application/x-www-form-urlencoded"
    },
    data={
        "grant_type": "client_credentials"
    }
)

response.raise_for_status()
token = response.json()["access_token"]
print("Token obtenido correctamente")


In [0]:
# FUNCIÓN PARA CONECTARSE A LA API

BASE_URL = "https://api.spotify.com/v1"

def spotify_get(endpoint, params=None):
    url = f"{BASE_URL}{endpoint}"

    headers = {
        "Authorization": f"Bearer {token}"
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code == 429:
        retry_after = int(
            response.headers.get("Retry-After", "5")
        )
        print(f"Rate limit. Esperando {retry_after} segundos...")
        time.sleep(retry_after)

        return spotify_get(endpoint, params)

    response.raise_for_status()
    return response.json()



In [0]:
genres = [
    "rock", "pop", "latin", "reggaeton", "hip-hop", "rap", "jazz", "classical",   "electronic", "metal", "indie", "alternative", "country", "r&b", "salsa",    "bachata", "merengue"
]

In [0]:
# FUNCION PARA EXTRAER LOS TRACKS
def search_tracks(query, limit=10, offset=0):

    return spotify_get("/search",
        params={
            "q": query,
            "type": "track",
            "limit": limit,
            "offset": offset
        }
    )

In [0]:
raw_records = []

for genre in genres:

    for offset in range(0, 100, 10):

        data = search_tracks(
            query=genre,
            limit=10,
            offset=offset
        )

        raw_records.append({
            "search_term": genre,
            "offset": offset,
            "extraction_timestamp": datetime.utcnow(),
            "payload": json.dumps(data)
        })

In [0]:
raw_df = spark.createDataFrame(raw_records)

display(raw_df)

In [0]:
(
    raw_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "spotify_catalog.raw.spotify_api_raw"
    )
)